# Machote de Machine Learning — Funciones Reutilizables
### Diplomado en Redes Neuronales y Deep Learning

---
Este notebook es una **librería de funciones reutilizables** para proyectos de ML.
Cada función tiene nombre descriptivo, para qué sirve, qué necesita y qué regresa.

**¿Cómo usarlo?**
1. Corre la Sección 0 (imports) y la Sección 1 (funciones) siempre al inicio.
2. En tu notebook de proyecto, copia las funciones que necesites.
3. Cada función tiene un ejemplo al final para que veas cómo se usa.

**Índice:**
- `Sección 0` — Imports (todo lo que se necesita instalar/importar)
- `Sección 1` — Cargar datos
- `Sección 2` — Explorar datos (EDA)
- `Sección 3` — Limpiar y preparar datos
- `Sección 4` — Dividir datos (train/test)
- `Sección 5` — Evaluar modelos
- `Sección 6` — Visualizar resultados

---
## Sección 0 — Imports
Corre esta celda siempre primero. Importa todo lo que las funciones de abajo necesitan.

In [ ]:
# ── Datos y matemáticas ──────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualización ────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── Machine Learning ─────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    mean_squared_error,
    r2_score
)
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# ── Datasets de práctica ─────────────────────────────────────────────
from sklearn.datasets import (
    load_breast_cancer,   # Clasificación binaria (maligno/benigno)
    load_iris,            # Clasificación multiclase (3 tipos de flor)
    load_wine,            # Clasificación multiclase (tipos de vino)
    load_diabetes,        # Regresión (predecir nivel de diabetes)
    load_boston           # Regresión (predecir precio de casas) — deprecated en sklearn nuevas
)

print("Imports listos")

---
## Sección 1 — Cargar datos

Hay tres formas comunes de obtener datos:
- **Desde sklearn** — datasets de práctica ya incluidos, sin descargar nada
- **Desde UCI** — repositorio oficial de datasets de ML
- **Desde un archivo CSV** — cuando tienes tus propios datos

Todas devuelven lo mismo al final: un DataFrame `X` con las características y un array `Y` con las etiquetas.

In [ ]:
# ════════════════════════════════════════════════════════════════════
# FUNCIÓN: cargar_desde_sklearn
# ────────────────────────────────────────────────────────────────────
# Para qué: cargar un dataset que ya viene dentro de scikit-learn.
#           No necesitas internet ni archivos.
#
# Parámetros:
#   dataset_func → la función de sklearn (load_breast_cancer, load_iris, etc.)
#
# Regresa:
#   X → DataFrame con las características (columnas = features)
#   Y → array numpy con las etiquetas (0, 1, 2...)
#   nombres_clases → lista con el nombre de cada clase
# ════════════════════════════════════════════════════════════════════
def cargar_desde_sklearn(dataset_func):
    dataset = dataset_func()
    X = pd.DataFrame(dataset.data, columns=dataset.feature_names)
    Y = dataset.target
    nombres_clases = list(dataset.target_names)

    print(f"Dataset cargado: {dataset_func.__name__}")
    print(f"  Muestras:        {X.shape[0]}")
    print(f"  Características: {X.shape[1]}")
    print(f"  Clases:          {nombres_clases}")

    return X, Y, nombres_clases


# ════════════════════════════════════════════════════════════════════
# FUNCIÓN: cargar_desde_uci
# ────────────────────────────────────────────────────────────────────
# Para qué: cargar un dataset desde UCI Machine Learning Repository.
#           Necesitas internet. No guarda archivos en tu máquina.
#
# Parámetros:
#   uci_id       → número de identificación del dataset en UCI
#   columna_y    → nombre de la columna que contiene el diagnóstico/etiqueta
#   mapeo_clases → diccionario para convertir texto a números
#                  Ejemplo: {'M': 0, 'B': 1}  o  {'Yes': 1, 'No': 0}
#                  Si ya son números, pon None.
#
# Regresa:
#   X → DataFrame con las características
#   Y → array numpy con las etiquetas numéricas
# ════════════════════════════════════════════════════════════════════
def cargar_desde_uci(uci_id, columna_y, mapeo_clases=None):
    from ucimlrepo import fetch_ucirepo
    dataset = fetch_ucirepo(id=uci_id)
    X = dataset.data.features
    Y_raw = dataset.data.targets

    if mapeo_clases is not None:
        Y = Y_raw[columna_y].map(mapeo_clases).to_numpy()
    else:
        Y = Y_raw[columna_y].to_numpy()

    print(f"Dataset UCI id={uci_id} cargado")
    print(f"  Muestras:        {X.shape[0]}")
    print(f"  Características: {X.shape[1]}")
    if mapeo_clases:
        print(f"  Mapeo de clases: {mapeo_clases}")

    return X, Y


# ════════════════════════════════════════════════════════════════════
# FUNCIÓN: cargar_desde_csv
# ────────────────────────────────────────────────────────────────────
# Para qué: cargar datos desde un archivo CSV que tienes en tu máquina
#           o desde una URL pública.
#
# Parámetros:
#   ruta      → ruta al archivo CSV, o URL completa
#   columna_y → nombre de la columna que contiene la etiqueta/resultado
#
# Regresa:
#   X → DataFrame con las características (todo excepto columna_y)
#   Y → Serie de pandas con las etiquetas
# ════════════════════════════════════════════════════════════════════
def cargar_desde_csv(ruta, columna_y):
    df = pd.read_csv(ruta)
    X = df.drop(columns=[columna_y])
    Y = df[columna_y]

    print(f"CSV cargado desde: {ruta}")
    print(f"  Muestras:        {X.shape[0]}")
    print(f"  Características: {X.shape[1]}")
    print(f"  Columna Y:       '{columna_y}'")

    return X, Y


# ── Ejemplos de uso ──────────────────────────────────────────────────
print("Funciones de carga definidas")
print()
print("Ejemplos de uso:")
print("  X, Y, clases = cargar_desde_sklearn(load_breast_cancer)")
print("  X, Y = cargar_desde_uci(17, 'Diagnosis', {'M': 0, 'B': 1})")
print("  X, Y = cargar_desde_csv('mis_datos.csv', 'resultado')")

---
## Sección 2 — Explorar datos (EDA)

EDA = Exploratory Data Analysis = "¿Qué tengo aquí?"

Antes de entrenar CUALQUIER modelo, siempre hay que explorar los datos:
- ¿Cuántos datos tenemos?
- ¿Hay valores faltantes?
- ¿Cómo se distribuyen los valores?
- ¿Qué características se relacionan más con lo que queremos predecir?

In [ ]:
# ════════════════════════════════════════════════════════════════════
# FUNCIÓN: resumen_rapido
# ────────────────────────────────────────────────────────────────────
# Para qué: ver un panorama completo del dataset de un vistazo.
#           Es lo primero que debes correr después de cargar datos.
#
# Parámetros:
#   X → DataFrame con las características
#   Y → array/Serie con las etiquetas
#
# Regresa: nada, solo imprime el resumen
# ════════════════════════════════════════════════════════════════════
def resumen_rapido(X, Y):
    print("═" * 50)
    print("RESUMEN DEL DATASET")
    print("═" * 50)
    print(f"Muestras (filas):         {X.shape[0]}")
    print(f"Características (cols):   {X.shape[1]}")
    print()

    # Valores faltantes
    faltantes = X.isnull().sum()
    cols_con_faltantes = faltantes[faltantes > 0]
    if len(cols_con_faltantes) == 0:
        print("Valores faltantes: ninguno")
    else:
        print(f"Valores faltantes en {len(cols_con_faltantes)} columnas:")
        print(cols_con_faltantes.to_string())
    print()

    # Distribución de clases
    print("Distribución de Y (etiquetas):")
    valores, conteos = np.unique(Y, return_counts=True)
    for v, c in zip(valores, conteos):
        barra = '█' * int(c / len(Y) * 30)
        print(f"  Clase {v}: {c:4d} ({c/len(Y)*100:.1f}%) {barra}")
    print()

    # Tipos de datos
    tipos = X.dtypes.value_counts()
    print("Tipos de datos en X:")
    for tipo, cantidad in tipos.items():
        print(f"  {str(tipo):<12}: {cantidad} columnas")
    print("═" * 50)


# ════════════════════════════════════════════════════════════════════
# FUNCIÓN: ver_correlacion_con_y
# ────────────────────────────────────────────────────────────────────
# Para qué: ver qué tan relacionada está cada característica con
#           lo que queremos predecir. Nos ayuda a elegir cuáles
#           características son útiles y cuáles son ruido.
#
# Cómo interpretar:
#   Correlación alta (abs > 0.5) → característica ÚTIL
#   Correlación baja (abs < 0.2) → probablemente ruido, considerar eliminar
#   Positiva (+) → cuando sube la característica, sube Y
#   Negativa (-) → cuando sube la característica, baja Y
#
# Parámetros:
#   X → DataFrame con las características
#   Y → array con las etiquetas
#   umbral → correlación mínima para resaltar (default 0.5)
#
# Regresa:
#   correlaciones → Serie de pandas ordenada de mayor a menor
# ════════════════════════════════════════════════════════════════════
def ver_correlacion_con_y(X, Y, umbral=0.5):
    df_temp = X.copy()
    df_temp['__Y__'] = Y
    correlaciones = df_temp.corr()['__Y__'].drop('__Y__').sort_values(key=abs, ascending=False)

    # Gráfica de barras horizontales
    fig, ax = plt.subplots(figsize=(8, max(4, len(correlaciones) * 0.3)))
    colors = ['#D85A30' if abs(v) >= umbral else '#888' for v in correlaciones]
    correlaciones.plot(kind='barh', ax=ax, color=colors, edgecolor='none')
    ax.axvline(x=0,      color='gray',  linewidth=0.8)
    ax.axvline(x=umbral,  color='green', linewidth=1.2, linestyle='--', label=f'umbral +{umbral}')
    ax.axvline(x=-umbral, color='green', linewidth=1.2, linestyle='--')
    ax.set_title(f'Correlación de cada característica con Y (umbral={umbral})')
    ax.set_xlabel('Correlación')
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f"\nCaracterísticas con |correlación| >= {umbral}:")
    utiles = correlaciones[correlaciones.abs() >= umbral]
    for feat, val in utiles.items():
        print(f"  {feat:<35} {val:+.4f}")
    print(f"\nTotal útiles: {len(utiles)} de {len(correlaciones)}")

    return correlaciones


# ════════════════════════════════════════════════════════════════════
# FUNCIÓN: ver_mapa_calor
# ────────────────────────────────────────────────────────────────────
# Para qué: ver las correlaciones entre todas las características
#           para detectar cuáles son redundantes entre sí.
#           Si dos columnas tienen correlación 1.0, dicen lo mismo
#           y puedes eliminar una.
#
# Parámetros:
#   X     → DataFrame con las características
#   Y     → array con las etiquetas (se agrega como columna extra)
#   top_n → cuántas características mostrar (las más correlacionadas con Y)
#             Si pones None, muestra todas.
#
# Regresa: nada, solo la gráfica
# ════════════════════════════════════════════════════════════════════
def ver_mapa_calor(X, Y, top_n=15):
    df_temp = X.copy()
    df_temp['diagnostico_Y'] = Y

    if top_n is not None and top_n < X.shape[1]:
        # Seleccionar las top_n más correlacionadas con Y
        corrs = df_temp.corr()['diagnostico_Y'].drop('diagnostico_Y').abs().sort_values(ascending=False)
        cols_top = corrs.head(top_n).index.tolist()
        df_temp = df_temp[cols_top + ['diagnostico_Y']]
        titulo = f'Mapa de calor — top {top_n} características + Y'
    else:
        titulo = 'Mapa de calor — todas las características + Y'

    fig, ax = plt.subplots(figsize=(max(8, len(df_temp.columns) * 0.7),
                                    max(6, len(df_temp.columns) * 0.7)))
    sns.heatmap(
        df_temp.corr(),
        annot=True, fmt='.2f',
        cmap='RdBu_r',
        vmin=-1, vmax=1,
        square=True,
        linewidths=0.3,
        ax=ax
    )
    ax.set_title(titulo, pad=15)
    plt.tight_layout()
    plt.show()


print("Funciones de exploración (EDA) definidas")
print()
print("Ejemplos de uso:")
print("  resumen_rapido(X, Y)")
print("  correlaciones = ver_correlacion_con_y(X, Y, umbral=0.5)")
print("  ver_mapa_calor(X, Y, top_n=15)")

---
## Sección 3 — Limpiar y preparar datos

Los datos del mundo real casi nunca llegan perfectos. Hay que:
- Seleccionar solo las características útiles
- Escalar los valores (para modelos más avanzados)
- Binarizar (para MPNeuron específicamente)

**Regla de oro:** siempre prepara train y test por separado, nunca juntos.
Si los mezclas, el test se contamina con información del train y el modelo hace trampa.

In [ ]:
# ════════════════════════════════════════════════════════════════════
# FUNCIÓN: seleccionar_por_correlacion
# ────────────────────────────────────────────────────────────────────
# Para qué: quedarnos solo con las características que tienen
#           correlación significativa con Y. Elimina el ruido.
#
# Parámetros:
#   X        → DataFrame completo
#   Y        → array con etiquetas
#   umbral   → correlación mínima (en valor absoluto) para conservar
#              0.5 es un buen punto de inicio. Sube si quedan muchas,
#              baja si quedan muy pocas.
#
# Regresa:
#   X_reducido          → DataFrame con solo las columnas útiles
#   features_elegidas   → lista de nombres de las columnas conservadas
# ════════════════════════════════════════════════════════════════════
def seleccionar_por_correlacion(X, Y, umbral=0.5):
    df_temp = X.copy()
    df_temp['__Y__'] = Y
    correlaciones = df_temp.corr()['__Y__'].drop('__Y__')
    features_elegidas = correlaciones[correlaciones.abs() >= umbral].index.tolist()

    print(f"Selección con umbral={umbral}:")
    print(f"  Conservadas: {len(features_elegidas)} de {X.shape[1]} características")
    print(f"  Eliminadas:  {X.shape[1] - len(features_elegidas)} características")

    return X[features_elegidas], features_elegidas


# ════════════════════════════════════════════════════════════════════
# FUNCIÓN: escalar_datos
# ────────────────────────────────────────────────────────────────────
# Para qué: poner todos los valores en la misma escala.
#           Muchos modelos (regresión logística, SVM, redes neuronales)
#           funcionan mucho mejor cuando las columnas no tienen rangos
#           muy diferentes entre sí.
#
# Ejemplo sin escalar: 'area' va de 143 a 2501, 'smoothness' de 0.05 a 0.16
#           El modelo le da más peso a 'area' solo por ser más grande.
# Ejemplo escalado: ambas van de 0 a 1 → el modelo las trata igual.
#
# Parámetros:
#   X_train → datos de entrenamiento
#   X_test  → datos de prueba
#   metodo  → 'standard' (media=0, std=1) o 'minmax' (0 a 1)
#              'standard' es el más común para modelos lineales y redes neuronales
#              'minmax' es útil cuando necesitas valores entre 0 y 1
#
# Regresa:
#   X_train_scaled, X_test_scaled → ambos escalados con la misma referencia
# ════════════════════════════════════════════════════════════════════
def escalar_datos(X_train, X_test, metodo='standard'):
    if metodo == 'standard':
        scaler = StandardScaler()   # Centra en 0, desviación estándar 1
    elif metodo == 'minmax':
        scaler = MinMaxScaler()     # Escala al rango [0, 1]
    else:
        raise ValueError("metodo debe ser 'standard' o 'minmax'")

    # IMPORTANTE: fit solo con train, transform en ambos
    # Si hicieras fit con test, el scaler aprendería info del test → trampa
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)      # usa la escala del train

    print(f"Escalado con método '{metodo}':")
    print(f"  X_train: shape {X_train_scaled.shape}")
    print(f"  X_test:  shape {X_test_scaled.shape}")

    return X_train_scaled, X_test_scaled


# ════════════════════════════════════════════════════════════════════
# FUNCIÓN: binarizar_datos
# ────────────────────────────────────────────────────────────────────
# Para qué: convertir valores continuos a 0 o 1.
#           Solo se necesita para la MPNeuron — los demás modelos
#           aceptan valores continuos directamente.
#
# ADVERTENCIA sobre los labels:
#   Si la correlación de tus features con Y es POSITIVA  → labels=[0, 1]
#   Si la correlación de tus features con Y es NEGATIVA  → labels=[1, 0]
#   Confundirlos invierte la lógica del modelo (como pasó en clase).
#
# Parámetros:
#   X_train, X_test → DataFrames a binarizar
#   correlaciones   → Serie devuelta por ver_correlacion_con_y()
#                     Se usa para detectar automáticamente la dirección.
#
# Regresa:
#   X_train_bin, X_test_bin → DataFrames binarizados
# ════════════════════════════════════════════════════════════════════
def binarizar_datos(X_train, X_test, correlaciones):
    # Detectamos la dirección dominante de las correlaciones
    corr_promedio = correlaciones.mean()

    if corr_promedio >= 0:
        # Correlación positiva → valor alto = Y alto → labels normales
        labels = [0, 1]
        print("Correlación promedio POSITIVA → labels=[0, 1] (valor bajo=0, alto=1)")
    else:
        # Correlación negativa → valor alto = Y bajo → labels invertidos
        labels = [1, 0]
        print("Correlación promedio NEGATIVA → labels=[1, 0] (valor bajo=1, alto=0)")

    X_train_bin = X_train.apply(pd.cut, bins=2, labels=labels)
    X_test_bin  = X_test.apply(pd.cut,  bins=2, labels=labels)

    print(f"Binarización lista: {X_train_bin.shape[1]} columnas")

    return X_train_bin, X_test_bin


print("Funciones de preparación de datos definidas")
print()
print("Ejemplos de uso:")
print("  X_red, features = seleccionar_por_correlacion(X, Y, umbral=0.5)")
print("  X_tr_sc, X_te_sc = escalar_datos(X_train, X_test, metodo='standard')")
print("  X_tr_bin, X_te_bin = binarizar_datos(X_train, X_test, correlaciones)")

---
## Sección 4 — Dividir datos (train / test)

Esta es la regla más importante en ML:
**nunca evalúes con los mismos datos con los que entrenaste.**

En proyectos más avanzados también se agrega un conjunto de **validación** (para ajustar hiperparámetros), pero por ahora train + test es suficiente.

In [ ]:
# ════════════════════════════════════════════════════════════════════
# FUNCIÓN: dividir_datos
# ────────────────────────────────────────────────────────────────────
# Para qué: separar los datos en dos grupos:
#   - Train (entrenamiento): el modelo aprende con estos
#   - Test (prueba): evaluamos qué tan bien aprendió con datos que nunca vio
#
# Parámetros:
#   X            → DataFrame con características
#   Y            → array con etiquetas
#   proporcion_test → qué fracción va al test (0.25 = 25% test, 75% train)
#   semilla      → número fijo para que la división sea siempre la misma
#                  (reproducible). Puedes poner cualquier número.
#   estratificar → True si Y es clasificación (mantiene proporciones de clases)
#                  False si Y es regresión (números continuos)
#
# Regresa:
#   X_train, X_test, y_train, y_test
# ════════════════════════════════════════════════════════════════════
def dividir_datos(X, Y, proporcion_test=0.25, semilla=42, estratificar=True):
    stratify = Y if estratificar else None

    X_train, X_test, y_train, y_test = train_test_split(
        X, Y,
        test_size=proporcion_test,
        random_state=semilla,
        stratify=stratify
    )

    print(f"División de datos (semilla={semilla}):")
    print(f"  Train: {len(X_train)} muestras ({(1-proporcion_test)*100:.0f}%)")
    print(f"  Test:  {len(X_test)} muestras ({proporcion_test*100:.0f}%)")

    if estratificar:
        print("  Distribución de clases en train:", dict(zip(*np.unique(y_train, return_counts=True))))
        print("  Distribución de clases en test: ", dict(zip(*np.unique(y_test, return_counts=True))))

    return X_train, X_test, y_train, y_test


print("Función de división definida")
print()
print("Ejemplo de uso:")
print("  X_train, X_test, y_train, y_test = dividir_datos(X, Y, proporcion_test=0.25)")

---
## Sección 5 — Evaluar modelos

Hay diferentes formas de medir qué tan bien funciona un modelo:

**Para clasificación** (predice categorías: maligno/benigno, perro/gato):
- `accuracy` → % de aciertos totales
- `precision` → de los que predije positivo, ¿cuántos realmente lo eran?
- `recall` → de todos los positivos reales, ¿cuántos encontré?
- `F1` → balance entre precision y recall

**Para regresión** (predice números: precio, temperatura, edad):
- `MSE` → error cuadrático medio (penaliza errores grandes)
- `RMSE` → raíz del MSE (mismas unidades que Y)
- `R²` → qué tan bien explica el modelo la variación de Y (0 a 1)

In [ ]:
# ════════════════════════════════════════════════════════════════════
# FUNCIÓN: evaluar_clasificacion
# ────────────────────────────────────────────────────────────────────
# Para qué: medir qué tan bien predice un modelo de clasificación.
#           Muestra accuracy, reporte completo y matriz de confusión.
#
# Parámetros:
#   y_real      → las etiquetas correctas (y_test)
#   y_predicho  → las predicciones del modelo
#   nombres_clases → lista de nombres ['Maligno', 'Benigno'] (opcional)
#
# Regresa:
#   acc → el accuracy como número (útil para comparar modelos)
# ════════════════════════════════════════════════════════════════════
def evaluar_clasificacion(y_real, y_predicho, nombres_clases=None):
    acc = accuracy_score(y_real, y_predicho)
    cm  = confusion_matrix(y_real, y_predicho)

    print(f"Accuracy: {acc:.4f} ({acc*100:.1f}%)")
    print()
    print("Reporte completo:")
    print(classification_report(y_real, y_predicho, target_names=nombres_clases))

    # Matriz de confusión visual
    etiquetas = nombres_clases if nombres_clases else [str(c) for c in np.unique(y_real)]
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=[f'Pred. {e}' for e in etiquetas],
        yticklabels=[f'Real {e}' for e in etiquetas],
        ax=ax
    )
    ax.set_title(f'Matriz de Confusión — Accuracy: {acc*100:.1f}%')
    plt.tight_layout()
    plt.show()

    return acc


# ════════════════════════════════════════════════════════════════════
# FUNCIÓN: evaluar_regresion
# ────────────────────────────────────────────────────────────────────
# Para qué: medir qué tan bien predice un modelo de regresión.
#           Se usa cuando Y son números continuos (precios, temperaturas).
#
# Parámetros:
#   y_real     → los valores reales (y_test)
#   y_predicho → las predicciones del modelo
#
# Regresa:
#   dict con mse, rmse y r2
# ════════════════════════════════════════════════════════════════════
def evaluar_regresion(y_real, y_predicho):
    mse  = mean_squared_error(y_real, y_predicho)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_real, y_predicho)

    print(f"MSE:  {mse:.4f}   (error cuadrático medio)")
    print(f"RMSE: {rmse:.4f}  (raíz del error — mismas unidades que Y)")
    print(f"R²:   {r2:.4f}   (1.0 = perfecto, 0.0 = no explica nada)")

    # Gráfica real vs predicho
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(y_real, y_predicho, alpha=0.5, color='#3B8BD4', s=20)
    lims = [min(y_real.min(), y_predicho.min()), max(y_real.max(), y_predicho.max())]
    ax.plot(lims, lims, 'r--', linewidth=1, label='Predicción perfecta')
    ax.set_xlabel('Valores reales')
    ax.set_ylabel('Valores predichos')
    ax.set_title(f'Real vs Predicho — R²: {r2:.4f}')
    ax.legend()
    plt.tight_layout()
    plt.show()

    return {'mse': mse, 'rmse': rmse, 'r2': r2}


print("Funciones de evaluación definidas")
print()
print("Ejemplos de uso:")
print("  acc = evaluar_clasificacion(y_test, Y_pred, ['Maligno', 'Benigno'])")
print("  metricas = evaluar_regresion(y_test, Y_pred)")

---
## Sección 6 — Visualizar datos

Funciones para graficar y entender los datos antes de modelar.

In [ ]:
# ════════════════════════════════════════════════════════════════════
# FUNCIÓN: ver_distribucion
# ────────────────────────────────────────────────────────────────────
# Para qué: ver cómo se distribuyen los valores de una columna.
#           Nos ayuda a entender si los datos están balanceados,
#           si hay valores extremos (outliers), etc.
#
# Parámetros:
#   X       → DataFrame
#   columna → nombre de la columna a graficar
#   Y       → array de etiquetas (opcional). Si se pone, colorea
#             la distribución por clase.
# ════════════════════════════════════════════════════════════════════
def ver_distribucion(X, columna, Y=None):
    fig, ax = plt.subplots(figsize=(7, 4))

    if Y is not None:
        for clase in np.unique(Y):
            valores = X[columna][Y == clase]
            ax.hist(valores, bins=30, alpha=0.6, label=f'Clase {clase}')
        ax.legend()
    else:
        ax.hist(X[columna], bins=30, color='#3B8BD4', alpha=0.8)

    ax.set_xlabel(columna)
    ax.set_ylabel('Frecuencia')
    ax.set_title(f'Distribución de: {columna}')
    plt.tight_layout()
    plt.show()


# ════════════════════════════════════════════════════════════════════
# FUNCIÓN: comparar_modelos
# ────────────────────────────────────────────────────────────────────
# Para qué: comparar el accuracy de varios modelos en una gráfica.
#           Muy útil cuando pruebas diferentes algoritmos.
#
# Parámetros:
#   resultados → diccionario {'NombreModelo': accuracy, ...}
#   Ejemplo:   {'MPNeuron': 0.91, 'Regresión Logística': 0.96}
# ════════════════════════════════════════════════════════════════════
def comparar_modelos(resultados):
    nombres = list(resultados.keys())
    valores = list(resultados.values())

    fig, ax = plt.subplots(figsize=(max(6, len(nombres) * 1.5), 4))
    colors = ['#D85A30' if v == max(valores) else '#3B8BD4' for v in valores]
    bars = ax.bar(nombres, [v * 100 for v in valores], color=colors, edgecolor='none')

    # Etiquetas encima de cada barra
    for bar, val in zip(bars, valores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val*100:.1f}%', ha='center', va='bottom', fontsize=11)

    ax.set_ylabel('Accuracy (%)')
    ax.set_ylim(0, 110)
    ax.set_title('Comparación de modelos')
    ax.axhline(y=max(valores)*100, color='orange', linestyle='--', alpha=0.5, linewidth=1)
    plt.tight_layout()
    plt.show()

    mejor = max(resultados, key=resultados.get)
    print(f"Mejor modelo: {mejor} con {resultados[mejor]*100:.1f}% de accuracy")


print("Funciones de visualización definidas")
print()
print("Ejemplos de uso:")
print("  ver_distribucion(X, 'mean radius', Y)")
print("  comparar_modelos({'MPNeuron': 0.91, 'Log. Regression': 0.96})")

---
## Sección 7 — Ejemplo completo usando el machote

Aquí vemos cómo se usan TODAS las funciones juntas en un proyecto real.
Este es el flujo estándar que vas a repetir en casi todos los ejercicios del diplomado.

In [ ]:
from sklearn.linear_model import LogisticRegression  # El modelo más simple después de MPNeuron

print("═" * 50)
print("FLUJO COMPLETO DE UN PROYECTO ML")
print("═" * 50)

# ── Paso 1: Cargar datos ─────────────────────────────────────────────
print("\n--- Paso 1: Cargar ---")
X, Y, clases = cargar_desde_sklearn(load_breast_cancer)

# ── Paso 2: Explorar ─────────────────────────────────────────────────
print("\n--- Paso 2: Explorar ---")
resumen_rapido(X, Y)

# ── Paso 3: Analizar correlaciones ───────────────────────────────────
print("\n--- Paso 3: Correlaciones ---")
correlaciones = ver_correlacion_con_y(X, Y, umbral=0.5)

# ── Paso 4: Seleccionar características útiles ───────────────────────
print("\n--- Paso 4: Selección ---")
X_reducido, features = seleccionar_por_correlacion(X, Y, umbral=0.5)

# ── Paso 5: Dividir en train y test ──────────────────────────────────
print("\n--- Paso 5: División ---")
X_train, X_test, y_train, y_test = dividir_datos(X_reducido, Y)

# ── Paso 6: Escalar los datos ────────────────────────────────────────
print("\n--- Paso 6: Escalar ---")
X_train_sc, X_test_sc = escalar_datos(X_train, X_test)

# ── Paso 7: Entrenar el modelo ───────────────────────────────────────
print("\n--- Paso 7: Entrenar ---")
modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_train_sc, y_train)
print("Modelo entrenado")

# ── Paso 8: Predecir y evaluar ───────────────────────────────────────
print("\n--- Paso 8: Evaluar ---")
Y_pred = modelo.predict(X_test_sc)
acc = evaluar_clasificacion(y_test, Y_pred, clases)

---
## Referencia rápida — ¿Cuándo uso qué?

| Situación | Función a usar |
|-----------|----------------|
| Acabo de cargar datos, ¿qué tengo? | `resumen_rapido(X, Y)` |
| ¿Qué columnas son útiles? | `ver_correlacion_con_y(X, Y)` |
| ¿Hay columnas redundantes entre sí? | `ver_mapa_calor(X, Y)` |
| Quiero quedarme solo con las útiles | `seleccionar_por_correlacion(X, Y, umbral=0.5)` |
| Voy a usar un modelo avanzado (no MPNeuron) | `escalar_datos(X_train, X_test)` |
| Voy a usar MPNeuron | `binarizar_datos(X_train, X_test, correlaciones)` |
| Separar datos para entrenar | `dividir_datos(X, Y)` |
| Medir qué tan bien clasifica | `evaluar_clasificacion(y_test, Y_pred)` |
| Medir qué tan bien regresiona | `evaluar_regresion(y_test, Y_pred)` |
| Comparar varios modelos | `comparar_modelos({'ModeloA': 0.91, 'ModeloB': 0.96})` |